# Practical 6: Keras MLP for Regression (California Housing)


## 1. Dataset Preparation


In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


In [ ]:
# Load and explore the dataset
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target  # MedHouseVal

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
X.head()


In [ ]:
# Basic statistics / exploration
X.describe().T


In [ ]:
print(housing.DESCR[:1200])


In [ ]:
# Check for missing values
print("Missing values per column:\n", X.isnull().sum())
print("\nTotal missing values:", X.isnull().sum().sum())


In [ ]:
# Target distribution
plt.figure(figsize=(6, 4))
sns.histplot(y, bins=40, kde=True)
plt.title("Distribution of Median House Value (Target)")
plt.xlabel("Median House Value ($100,000s)")
plt.tight_layout()
plt.show()

print("Target summary statistics:\n", y.describe())


In [ ]:
# Feature distributions / scale check
X.hist(figsize=(14, 8), bins=30)
plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.show()


In [ ]:
# Train / test split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.2, random_state=SEED
)

print("Train shape:", X_train_raw.shape, " Test shape:", X_test_raw.shape)


In [ ]:
# Feature scaling: fit scaler ONLY on training data, then transform train & test
# (prevents data leakage from the test set into preprocessing)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("Post-scaling mean (train, approx 0):", X_train.mean(axis=0).round(2))
print("Post-scaling std (train, approx 1):", X_train.std(axis=0).round(2))


## 2. MLP Model Development


In [ ]:
def build_mlp(input_dim, hidden_activation="relu", loss_function="mse"):
    """Builds and compiles an MLP for regression with 2 hidden layers."""
    model = Sequential([
        Dense(64, activation=hidden_activation, input_shape=(input_dim,), name="hidden_1"),
        Dense(32, activation=hidden_activation, name="hidden_2"),
        Dense(1, activation="linear", name="output"),
    ])
    model.compile(
        optimizer="adam",
        loss=loss_function,
        metrics=["mae", "mse"],
    )
    return model

# Quick summary for reporting: architecture with default settings (ReLU + MSE)
demo_model = build_mlp(X_train.shape[1], "relu", "mse")
demo_model.summary()


## 4. Experiment with Loss Functions


In [ ]:
EPOCHS = 100
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2

results = {}      # stores final metrics for the comparison table
histories = {}     # stores training history for plotting
models_dict = {}   # stores trained models

def run_experiment(name, hidden_activation, loss_function):
    print(f"\n=== Running: {name} (activation={hidden_activation}, loss={loss_function}) ===")
    tf.random.set_seed(SEED)
    model = build_mlp(X_train.shape[1], hidden_activation, loss_function)

    history = model.fit(
        X_train, y_train,
        validation_split=VALIDATION_SPLIT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    # Evaluate on the held-out test set
    y_pred = model.predict(X_test, verbose=0).flatten()

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results[name] = {
        "activation": hidden_activation,
        "loss_function": loss_function,
        "train_loss": history.history["loss"][-1],
        "val_loss": history.history["val_loss"][-1],
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }
    histories[name] = history
    models_dict[name] = model

    print(f"Test RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")
    return model, history


### Running Experiment A - Activation Functions (loss fixed to MSE)


In [ ]:
run_experiment("Model 1 (ReLU, MSE)",    "relu",    "mse")
run_experiment("Model 2 (Sigmoid, MSE)", "sigmoid", "mse")
run_experiment("Model 3 (Tanh, MSE)",    "tanh",    "mse")


### Running Experiment B - Loss Functions (activation fixed to ReLU)


In [ ]:
run_experiment("Model 4 (ReLU, MAE)",   "relu", "mae")
run_experiment("Model 5 (ReLU, Huber)", "relu", "huber")


## 5. Model Training - Loss Curves


In [ ]:
def plot_loss_curve(name, history):
    hist = history.history
    plt.figure(figsize=(6, 4))
    plt.plot(hist["loss"], label="Train Loss")
    plt.plot(hist["val_loss"], label="Validation Loss")
    plt.title(f"{name}\nLoss vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

for name, history in histories.items():
    plot_loss_curve(name, history)


In [ ]:
# Overlay comparison: validation loss across the 3 activation-function models (MSE loss)
plt.figure(figsize=(8, 5))
for name in ["Model 1 (ReLU, MSE)", "Model 2 (Sigmoid, MSE)", "Model 3 (Tanh, MSE)"]:
    plt.plot(histories[name].history["val_loss"], label=name)
plt.title("Validation Loss - Activation Function Comparison (loss=MSE)")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss (MSE)")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Model Evaluation


In [ ]:
# Predicted vs. Actual scatter plots for all 5 models
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, (name, model) in enumerate(models_dict.items()):
    y_pred = model.predict(X_test, verbose=0).flatten()
    axes[idx].scatter(y_test, y_pred, alpha=0.3, s=10)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[idx].plot(lims, lims, 'r--', linewidth=1)
    axes[idx].set_title(name, fontsize=9)
    axes[idx].set_xlabel("Actual")
    axes[idx].set_ylabel("Predicted")

for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## 7. Comparison Table


In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df[["activation", "loss_function", "MAE", "RMSE", "R2",
                                "MSE", "train_loss", "val_loss"]]
comparison_df = comparison_df.round(4)
comparison_df


In [ ]:
# Save the comparison table
comparison_df.to_csv("regression_model_comparison_results.csv")
print("Saved to regression_model_comparison_results.csv")
